# Phase 1 — Exploring the feature table

**Exit criterion:** `features.parquet` exists and this notebook renders Mumbai's heat map.

This reads the assembled feature table (`data/processed/features.parquet`) — one row per
~200 m cell, 42 columns — and does no Earth Engine work, so it runs in seconds. It is the
first look at the data the Phase 2 model will learn from: the surface-heat map, the vegetation
inverse, what correlates with heat, and the ward-level summary a planner would read.

Everything here is *surface* temperature at ~10:30 overpass (ADR-0005), and the caveats logged
during Phase 1 — the albedo confound, OSM under-mapping, weather being near-noise — are
carried into how the plots are read.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURES = REPO_ROOT / "data" / "processed" / "features.parquet"
if not FEATURES.exists():
    raise FileNotFoundError(
        f"{FEATURES} not found — build it first:  uv run python -m data_pipeline.assemble"
    )

gdf = gpd.read_parquet(FEATURES)
wards = gpd.read_file(REPO_ROOT / "data" / "processed" / "wards.geojson")

# Project to UTM 43N so map shapes and distances are true (docs/conventions.md).
gdf_m = gdf.to_crs(32643)
wards_m = wards.to_crs(32643)

# Land cells only for statistics — cells that are mostly sea carry water temperature, not
# urban heat (the land_fraction caveat in the data dictionary).
land = gdf[gdf["land_fraction"] >= 0.5]

print(f"{len(gdf):,} cells × {gdf.shape[1]} columns   ({len(land):,} land cells)")
print(f"LST  min {gdf['lst_mean'].min():.1f}  mean {gdf['lst_mean'].mean():.1f}  "
      f"max {gdf['lst_mean'].max():.1f} °C")

## 1. The heat map — surface temperature across Mumbai

The headline. Brighter is hotter. The colour scale is clipped to the 2nd–98th percentile so a
handful of extreme cells don't wash out the gradient, and ward outlines are drawn for
orientation. `inferno` is a perceptually-uniform, colour-blind-safe ramp — equal steps in
temperature look like equal steps in colour, which a rainbow palette cannot promise.

In [ ]:
def choropleth(ax, column, cmap, title, label, *, pct=(2, 98), vcenter=None):
    """Plot one feature as a cell choropleth with ward outlines."""
    vals = gdf_m[column]
    vmin, vmax = np.percentile(vals, pct)
    norm = None
    if vcenter is not None:
        from matplotlib.colors import TwoSlopeNorm
        m = max(abs(vmin), abs(vmax))
        norm = TwoSlopeNorm(vmin=-m, vcenter=vcenter, vmax=m)
        vmin = vmax = None
    gdf_m.plot(column=column, cmap=cmap, ax=ax, vmin=vmin, vmax=vmax, norm=norm,
               linewidth=0, antialiased=False)
    wards_m.boundary.plot(ax=ax, color="white", linewidth=0.4, alpha=0.6)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    sm = plt.cm.ScalarMappable(cmap=cmap,
                               norm=norm or plt.Normalize(vmin=vmin, vmax=vmax))
    cbar = ax.figure.colorbar(sm, ax=ax, shrink=0.6, pad=0.01)
    cbar.set_label(label)


fig, ax = plt.subplots(figsize=(9, 10))
choropleth(ax, "lst_mean", "inferno",
           "Mumbai surface temperature — dry-season median, ~200 m cells",
           "Land surface temperature (°C)")
fig.tight_layout()
plt.show()

## 2. The vegetation inverse

The project's premise is that vegetation cools and built-up warms. Put the heat map beside
NDVI and the inverse is visible by eye: the green north (Sanjay Gandhi National Park, Aarey)
is cool; the dense grey core and the eastern industrial belt are hot. Two different
satellites — Landsat thermal and Sentinel-2 optical — telling the same story.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 9))
choropleth(axes[0], "lst_mean", "inferno", "Surface temperature", "LST (°C)")
choropleth(axes[1], "ndvi_mean", "YlGn", "Vegetation (NDVI)", "NDVI")
fig.suptitle("Hot where grey, cool where green", fontsize=14, y=0.98)
fig.tight_layout()
plt.show()

## 3. What correlates with heat

Univariate correlation of every numeric feature with `lst_mean`, over land cells. This is the
whole of Phase 1 in one chart, and it is the first sanity check the Phase 2 model must not
contradict. Bars are coloured by sign — warm (red) drives heat up, cool (blue) down.

Read the flags logged during the build straight off it:

- **`ndbi_mean` leads** (+0.74) — built-up index is the strongest single driver.
- **`albedo` sits in the *warm* group** (~+0.67) — the **confound**: brighter reads hotter here
  because dark water is cool and bright bare ground is hot. The cool-roof lever must use a
  cited coefficient, never this correlation (`ml-methodology.md` §6).
- **Weather is flat** (~±0.02) — no within-city signal; a Phase 2 drop candidate.
- **`built_neigh_mean` ≈ `built_fraction`** — the neighbourhood carries as much signal as the
  cell: spatial autocorrelation, hence spatial block CV (ADR-0006).

In [ ]:
drop = {"cell_id", "grid_row", "grid_col", "lst_mean", "lst_p90", "lst_obs_count", "wc_pixels"}
num = land.select_dtypes("number").drop(columns=[c for c in drop if c in land], errors="ignore")
corr = num.corrwith(land["lst_mean"]).sort_values()

fig, ax = plt.subplots(figsize=(9, 11))
colors = ["#c0392b" if v > 0 else "#2c6fbb" for v in corr]  # warm = up, cool = down
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="#444", linewidth=0.8)
ax.set_xlabel("Correlation with surface temperature (land cells)")
ax.set_title("Drivers of surface heat — univariate correlation with LST", fontsize=13)
ax.grid(axis="x", color="0.9")
ax.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()

### 3.1 Correlation among the main drivers

A compact matrix of the load-bearing features, to see how they relate to each other (not just
to LST). Diverging `RdBu_r` centred at zero: red pairs move together, blue move oppositely.
Useful for spotting redundancy — e.g. `built_fraction`, `ndbi_mean` and `impervious_fraction`
are near-collinear, which SHAP will need to share credit across in Phase 2.

In [ ]:
keys = ["lst_mean", "ndbi_mean", "built_fraction", "impervious_fraction", "albedo",
        "pop_density", "ndvi_mean", "tree_fraction", "water_fraction", "dist_coast",
        "elevation_mean", "air_temp_mean"]
cm = land[keys].corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(keys)), keys, rotation=90, fontsize=9)
ax.set_yticks(range(len(keys)), keys, fontsize=9)
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f"{cm.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(cm.iloc[i, j]) > 0.55 else "black", fontsize=7)
ax.figure.colorbar(im, ax=ax, shrink=0.7, label="Pearson r")
ax.set_title("Correlation among the main drivers", fontsize=13)
fig.tight_layout()
plt.show()

## 4. Ward summary

The unit a planner acts in. Mean surface temperature, greenness, density and built fraction
per BMC ward (land cells), ranked hottest first. This is the raw material of the Phase 2
hotspot ranking and Heat Vulnerability Index — though the HVI will weigh heat *against*
population and lack of green, so the final priority order is not identical to this table.

In [ ]:
ward = (
    land.groupby("ward_code")
    .agg(
        cells=("cell_id", "size"),
        lst=("lst_mean", "mean"),
        ndvi=("ndvi_mean", "mean"),
        built=("built_fraction", "mean"),
        pop_density=("pop_density", "mean"),
    )
    .sort_values("lst", ascending=False)
    .round(2)
)
print(ward.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
order = ward.sort_values("lst")
norm = plt.Normalize(order["lst"].min(), order["lst"].max())
sm = plt.cm.ScalarMappable(cmap="inferno", norm=norm)
ax.barh(order.index, order["lst"], color=[sm.to_rgba(v) for v in order["lst"]])
ax.set_xlabel("Mean surface temperature (°C)")
ax.set_ylabel("BMC ward")
ax.set_xlim(order["lst"].min() - 1, order["lst"].max() + 0.5)
ax.set_title("Ward ranking by mean surface temperature", fontsize=13)
ax.grid(axis="x", color="0.9")
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()

## 5. Where heat and people overlap — the vulnerability seed

Surface heat only matters where people are. Plotting population density against LST, the cells
in the top decile of *both* — hot **and** dense — are the ones a heat action plan targets.
This is the intuition the Heat Vulnerability Index formalises in Phase 2; here it is visible
in the raw data.

In [ ]:
hot = land["lst_mean"] > land["lst_mean"].quantile(0.9)
dense = land["pop_density"] > land["pop_density"].quantile(0.9)
both = hot & dense

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(land.loc[~both, "lst_mean"], land.loc[~both, "pop_density"],
           s=6, color="0.75", alpha=0.5, label="other cells")
ax.scatter(land.loc[both, "lst_mean"], land.loc[both, "pop_density"],
           s=14, color="#c0392b", alpha=0.8, label="hot AND dense (top decile of both)")
ax.set_xlabel("Surface temperature (°C)")
ax.set_ylabel("Population density (persons/km²)")
ax.set_title(f"The vulnerability seed — {int(both.sum())} cells both hot and dense", fontsize=12)
ax.legend(frameon=False)
ax.grid(color="0.92")
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()

print("wards holding the most hot-and-dense cells:")
print(land.loc[both, "ward_code"].value_counts().head(6).to_string())

## What Phase 1 established

- A validated feature table: **11,944 cells × 42 columns**, one row per ~200 m cell, zero
  nulls, every column inside its physical range.
- The core premise holds in the data: vegetation and water cool, built-up and population warm,
  and two independent satellites agree.
- The heat is where the people are — the basis for a meaningful vulnerability index.

## What Phase 2 must respect

- **Spatial block cross-validation**, not a random split — the neighbourhood features correlate
  with LST almost as strongly as the cell's own, so a random split leaks (ADR-0006).
- **The albedo confound** — its correlation with LST is the wrong sign; the cool-roof lever
  uses a cited coefficient, and a positive albedo SHAP is expected, not a bug.
- **Feature redundancy** — built/NDBI/impervious are near-collinear; SHAP shares credit.
- **Weather is a drop candidate** — near-zero within-city signal.
- Every output is *surface* temperature, mid-morning, dry-season (ADR-0005).